# DAC Latent Encoder

Encodes audio files to DAC latent `.pt` files given an input and output directory.

- Output shape: `(1, T, 64)` — float32
- Mirrors input folder structure in output directory
- Resumable — skips already-encoded files

In [ ]:
!pip install descript-audio-codec descript-audiotools librosa -q

In [1]:
import gc
import json
import sys
import warnings
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn.utils as nn_utils
from torch.nn.utils import parametrizations

# Force new weight_norm API for dependencies that still call the deprecated helper
nn_utils.weight_norm = parametrizations.weight_norm

import dac
from audiotools import AudioSignal
import torch.nn as nn

warnings.filterwarnings('ignore')

print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch:        2.9.0+cpu
CUDA available: False


In [2]:



class DACLatentProcessor:
    """
    Encodes audio files to DAC latent representations.

    Output format:
        z      : (B, T, 768)  float16  — projected latents for DiT
        codes  : (B, num_codebooks, T)
        latents: pre-quantisation latents
    """

    def __init__(
        self,
        model_type: str = "44khz",
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        n_quantizers: Optional[int] = None,
        latent_dim: int = 768,
    ):
        self.model_type = model_type
        self.device = device
        self.n_quantizers = n_quantizers
        self.latent_dim = latent_dim

        print(f"Downloading/loading DAC model ({model_type})...")
        model_path = dac.utils.download(model_type=model_type)
        self.model = dac.DAC.load(model_path)
        if self.device.startswith("cuda") and not torch.cuda.is_available():
            print("CUDA requested but not available — falling back to CPU.")
            self.device = "cpu"
        self.model.to(self.device).eval()

        # 1024 = DAC 44khz native dim, 768 = target dim
        self.projection = nn.Linear(1024, self.latent_dim, bias=False).to(self.device)
        self.projection.eval()
        print(f"DAC model loaded on {self.device}")
        print(f"Projection: 1024 → {self.latent_dim}\n")

    def encode_file(self, audio_path: Path) -> dict:
        """Encode a single audio file and return latent tensors on CPU."""
        signal = AudioSignal(str(audio_path))
        signal = signal.to_mono()
        with torch.no_grad():
            signal = signal.to(self.device)
            x = self.model.preprocess(signal.audio_data, signal.sample_rate)
            z, codes, latents, _, _ = self.model.encode(x, self.n_quantizers)
            z = z.transpose(1, 2)               # (B, D, T) → (B, T, 1024)
            z = self.projection(z).half()       # (B, T, 1024) → (B, T, 768) float16
        return {
            "z":               z.cpu(),
            "codes":           codes.cpu(),
            "latents":         latents.cpu(),
            "sample_rate":     signal.sample_rate,
            "original_length": signal.signal_length,
        }

    def decode_latents(self, z: torch.Tensor, input_format: str = "TxD") -> np.ndarray:
        """
        Decode latents back to audio waveform.
        Note: expects raw DAC latents (D=1024), not projected ones.
        Use data["latents"] from the .pt file, not data["z"].
        """
        with torch.no_grad():
            z = z.to(self.device)
            if input_format == "TxD":
                z = z.transpose(1, 2)
            elif input_format != "DxT":
                raise ValueError(f"Unknown input_format '{input_format}'. Use 'TxD' or 'DxT'.")
            audio = self.model.decode(z)
        return audio.cpu().numpy()

    def load_latents(self, latent_path: str) -> dict:
        """Load a saved .pt latent file."""
        data = torch.load(latent_path, map_location="cpu")
        return {
            "z":               data["z"],
            "codes":           data["codes"],
            "latents":         data["latents"],
            "sample_rate":     int(data["sample_rate"]),
            "original_length": int(data["original_length"]),
        }

    def unload(self):
        """Free GPU memory."""
        if getattr(self, "model", None) is not None:
            del self.model
            self.model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        print("DAC model unloaded.")

print("DACLatentProcessor class defined.")


DACLatentProcessor class defined.


## Configuration

In [3]:
INPUT_DIR  = Path(r"C:\Users\Dhanuja\Downloads\dataset\dataset\fkdis")    # ← folder containing audio files
OUTPUT_DIR = Path(r"C:\Users\Dhanuja\Downloads\vibe")  # ← where .pt files will be saved

AUDIO_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
RESUME_MODE      = True   # set False to re-encode everything

print(f"Input:   {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Device:  {DEVICE}")
if not INPUT_DIR.exists():
    print(f"⚠ INPUT_DIR does not exist: {INPUT_DIR}")

Input:   C:\Users\Dhanuja\Downloads\dataset\dataset\fkdis
Output:  C:\Users\Dhanuja\Downloads\vibe
Device:  cpu


## Encode

In [4]:
import os
if STATE_FILE.exists():
    os.remove(STATE_FILE)
    print("State file deleted, will start fresh")

NameError: name 'STATE_FILE' is not defined

In [5]:
processor = DACLatentProcessor(model_type="44khz", device=DEVICE)

# Scan for audio files
print(f"Scanning {INPUT_DIR} ...")
audio_files = sorted(
    f for ext in AUDIO_EXTENSIONS for f in INPUT_DIR.glob(f"**/*{ext}")
)
print(f"Found {len(audio_files)} audio files\n")

# Resume: load existing state
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = OUTPUT_DIR / ".encoding_state.json"
processed_files_dict: dict = {}

if RESUME_MODE and STATE_FILE.exists():
    with open(STATE_FILE, "r") as f:
        processed_files_dict = json.load(f)
    print(f"Loaded resume state: {len(processed_files_dict)} entries")

if RESUME_MODE and processed_files_dict:
    audio_files = [
        f for f in audio_files
        if str(f.relative_to(INPUT_DIR)) not in processed_files_dict
        or processed_files_dict[str(f.relative_to(INPUT_DIR))] != "completed"
    ]
    skipped = sum(1 for v in processed_files_dict.values() if v == "completed")
    print(f"Already done: {skipped}  |  To process: {len(audio_files)}")
else:
    print(f"Will process all {len(audio_files)} files")

print()
for f in audio_files[:5]:
    print(f"  {f.name}")
if len(audio_files) > 5:
    print(f"  ... and {len(audio_files) - 5} more")


Downloading/loading DAC model (44khz)...
DAC model loaded on cpu
Projection: 1024 → 768

Scanning C:\Users\Dhanuja\Downloads\dataset\dataset\fkdis ...
Found 1 audio files

Will process all 1 files

  02035_Spowder_Miracle_Grow_instrumental_chunk013_26000ms_synth.mp3


In [6]:
processed_files: list = []   # list of (name, shape)
failed_files:    list = []   # list of (name, error)

print(f"Encoding {len(audio_files)} files  →  {OUTPUT_DIR}")
print("=" * 70)

for idx, audio_file in enumerate(audio_files, 1):
    # Compute rel_path here so it's available in both try and except
    rel_path = audio_file.relative_to(INPUT_DIR)
    file_key = str(rel_path)
    try:
        latent_data = processor.encode_file(audio_file)
        z = latent_data["z"]   # (1, T, 64)

        # Mirror directory structure from INPUT_DIR
        out_file = OUTPUT_DIR / rel_path.with_suffix(".pt")
        out_file.parent.mkdir(parents=True, exist_ok=True)

        torch.save(
            {
                "z":               z,
                "codes":           latent_data["codes"],
                "latents":         latent_data["latents"],
                "sample_rate":     latent_data["sample_rate"],
                "original_length": latent_data["original_length"],
            },
            out_file,
        )

        processed_files.append((audio_file.name, z.shape))
        processed_files_dict[file_key] = "completed"
        print(f"[{idx:>5}/{len(audio_files)}] {audio_file.name:<50} z={z.shape}")

    except Exception as e:
        failed_files.append((audio_file.name, str(e)))
        processed_files_dict[file_key] = "failed"
        print(f"[{idx:>5}/{len(audio_files)}] {audio_file.name:<50} ERROR: {str(e)[:60]}")

    # Persist state after every file (safe to interrupt)
    with open(STATE_FILE, "w") as f:
        json.dump(processed_files_dict, f, indent=2)

print("=" * 70)
print(f"Done.  Successful: {len(processed_files)}   Failed: {len(failed_files)}")

if failed_files:
    print("\nFailed files:")
    for name, err in failed_files:
        print(f"  {name}: {err}")


Encoding 1 files  →  C:\Users\Dhanuja\Downloads\vibe
[    1/1] 02035_Spowder_Miracle_Grow_instrumental_chunk013_26000ms_synth.mp3 z=torch.Size([1, 567, 768])
Done.  Successful: 1   Failed: 0


## Validate

In [ ]:
sample_pt_files = list(OUTPUT_DIR.glob("**/*.pt"))[:3]

if not sample_pt_files:
    print("No .pt files found in output directory.")
else:
    print("Format Validation")
    print("=" * 70)
    all_ok = True
    for pt_file in sample_pt_files:
        data = torch.load(pt_file, map_location="cpu", weights_only=False)
        z = data["z"]
        checks = {
            "shape (1,T,768)": z.ndim == 3 and z.shape[0] == 1 and z.shape[2] == 768,  # ← changed
            "float32":         z.dtype == torch.float32,
            "no NaN":          not torch.isnan(z).any().item(),
            "no Inf":          not torch.isinf(z).any().item(),
        }
        ok = all(checks.values())
        all_ok = all_ok and ok
        tick = "✓" if ok else "✗"
        print(f"{tick} {pt_file.name}")
        print(f"    shape={z.shape}  dtype={z.dtype}  "
              f"min={z.min():.4f}  max={z.max():.4f}  mean={z.mean():.4f}")
        for k, v in checks.items():
            print(f"    {'✓' if v else '✗'}  {k}")
        print()
    print("=" * 70)
    print("All checks passed ✓" if all_ok else "⚠ Some checks failed — see above.")

Format Validation
✗ 02035_Spowder_Miracle_Grow_instrumental_chunk013_26000ms_synth.pt
    shape=torch.Size([1, 567, 768])  dtype=torch.float16  min=-9.9688  max=9.7812  mean=-0.0242
    ✓  shape (1,T,768)
    ✗  float32
    ✓  no NaN
    ✓  no Inf

⚠ Some checks failed — see above.


: 

## Summary

In [15]:
pt_files = list(OUTPUT_DIR.glob("**/*.pt"))
total_mb = sum(f.stat().st_size for f in pt_files) / 1024**2

print("Latent Encoding Summary")
print("=" * 70)
print(f"Successfully encoded:  {len(processed_files)}")
print(f"Failed:                {len(failed_files)}")
print(f"Total .pt files:       {len(pt_files)}")
print(f"Total size on disk:    {total_mb:.2f} MB")
print(f"Input directory:       {INPUT_DIR}")
print(f"Output directory:      {OUTPUT_DIR}")

if processed_files:
    t_vals = [s[1] for _, s in processed_files]
    hop, sr = 512, 44100
    print()
    print("Time-frame (T) statistics:")
    print(f"  Min:    {min(t_vals)} frames  ({min(t_vals)*hop/sr:.2f}s)")
    print(f"  Max:    {max(t_vals)} frames  ({max(t_vals)*hop/sr:.2f}s)")
    print(f"  Mean:   {np.mean(t_vals):.1f} frames  ({np.mean(t_vals)*hop/sr:.2f}s)")
    print(f"  Median: {np.median(t_vals):.1f} frames  ({np.median(t_vals)*hop/sr:.2f}s)")

print("=" * 70)
print(f"\nLatents ready.  Point your dataloader at:\n  {OUTPUT_DIR}")

processor.unload()

Latent Encoding Summary
Successfully encoded:  28496
Failed:                1
Total .pt files:       28496
Total size on disk:    28603.89 MB
Input directory:       /workspace/vibe/source
Output directory:      /workspace/vibe/source_latents

Time-frame (T) statistics:
  Min:    173 frames  (2.01s)
  Max:    570 frames  (6.62s)
  Mean:   553.5 frames  (6.43s)
  Median: 567.0 frames  (6.58s)

Latents ready.  Point your dataloader at:
  /workspace/vibe/source_latents
DAC model unloaded.


## Decode a Saved Latent (Optional)

In [ ]:
from IPython.display import Audio, display

# ── Load ──────────────────────────────────────────────────────────────────────
sample_pt = list(OUTPUT_DIR.glob("**/*.pt"))[0]
data = torch.load(sample_pt, map_location="cpu", weights_only=False)

z = data["z"]
print(f"File:            {sample_pt.name}")
print(f"Latent shape:    {z.shape}  (should be (1, T, 64))")
print(f"Dtype:           {z.dtype}  (should be torch.float32)")
print(f"Range:           [{z.min():.4f}, {z.max():.4f}]")
print(f"Sample rate:     {data['sample_rate']} Hz")
print(f"Original length: {data['original_length']} samples")

# ── Decode back to audio (optional — needs GPU/CPU with enough memory) ────────
DECODE = True   # set False to skip

if DECODE:
    # Re-load DAC just for decoding (skip if processor is still loaded above)
    dec_processor = DACLatentProcessor(model_type="44khz", device=DEVICE)
    waveform = dec_processor.decode_latents(z, input_format="TxD")   # (1, 1, samples)
    audio_np  = waveform[0, 0]   # (samples,)
    sr        = int(data["sample_rate"])
    print(f"\nDecoded waveform shape: {audio_np.shape}  duration: {len(audio_np)/sr:.2f}s")
    display(Audio(audio_np, rate=sr))
    dec_processor.unload()